### **Target inicial:** ICU mortality (unitdischargestatus).

### **Prediction window:** primeiras 24h da UCI.

---

#### Esquema inicial dos agentes:

In [1]:
"""
                       eICU PATIENT
                            │
               DATA FROM FIRST 24 HOURS
                            │
        ┌───────────────────┼───────────────────┐
        ↓                   ↓                   ↓
   VITAL SIGNS          LABORATORY          CLINICAL
      AGENT                AGENT              AGENT
        │                   │                   │
        │                   │                   │
 HR, SpO₂, RR          Creatinine,          Demographics
 trends, etc.          WBC, Na, K...         Diagnoses
                                              History
        │                   │                   │
        └───────────────────┼───────────────────┘
                            │
                            ↓
                      SEVERITY AGENT
                   APACHE-related data
                            │
                 ┌──────────┴──────────┐
                 ↓                     ↓
             ML MODEL                 SHAP
                 │                     │
                 └──────────┬──────────┘
                            ↓
                    EXPLANATION AGENT
                            │
                            ↓
                       CRITIC AGENT
                         ↙       ↘
                      ERROR       OK
                        ↓          ↓
                      RETRY    FINAL REPORT
                                    ↓
                                  HUMAN
"""

'\n                       eICU PATIENT\n                            │\n               DATA FROM FIRST 24 HOURS\n                            │\n        ┌───────────────────┼───────────────────┐\n        ↓                   ↓                   ↓\n   VITAL SIGNS          LABORATORY          CLINICAL\n      AGENT                AGENT              AGENT\n        │                   │                   │\n        │                   │                   │\n HR, SpO₂, RR          Creatinine,          Demographics\n trends, etc.          WBC, Na, K...         Diagnoses\n                                              History\n        │                   │                   │\n        └───────────────────┼───────────────────┘\n                            │\n                            ↓\n                      SEVERITY AGENT\n                   APACHE-related data\n                            │\n                 ┌──────────┴──────────┐\n                 ↓                     ↓\n             ML MODE

#### Variávies (iniciais) a usar em cada *Data Agent*:

In [2]:
"""
                     PATIENT FIRST 24H
                            │
       ┌────────────────────┼────────────────────┐
       │                    │                    │
       ▼                    ▼                    ▼
 VITAL SIGNS AGENT    LABORATORY AGENT     CLINICAL AGENT

 HR                   Creatinine            Age
 SpO₂                 BUN                   Gender
 Respiration          Sodium                Unit type
 Temperature          Potassium             Admission source
 Systolic BP          Glucose               Admission diagnosis
 Mean BP              WBC                   0–24h diagnoses
                      Hemoglobin            Past history
                      Platelets
                      Bicarbonate
                      pH
                      ± Lactate
"""

'\n                     PATIENT FIRST 24H\n                            │\n       ┌────────────────────┼────────────────────┐\n       │                    │                    │\n       ▼                    ▼                    ▼\n VITAL SIGNS AGENT    LABORATORY AGENT     CLINICAL AGENT\n\n HR                   Creatinine            Age\n SpO₂                 BUN                   Gender\n Respiration          Sodium                Unit type\n Temperature          Potassium             Admission source\n Systolic BP          Glucose               Admission diagnosis\n Mean BP              WBC                   0–24h diagnoses\n                      Hemoglobin            Past history\n                      Platelets\n                      Bicarbonate\n                      pH\n                      ± Lactate\n'

#### Desenvolvimento sobre o funcionamento do *ML/Predictive Agent*:

In [3]:
"""
                         SEVERITY AGENT
                                │
                                │ receives outputs from
                                │ the 3 specialised agents
                                │
          ┌─────────────────────┼─────────────────────┐
          │                     │                     │
          ▼                     ▼                     ▼
     VITAL OUTPUT          LAB OUTPUT          CLINICAL OUTPUT

   HR, SpO₂, RR,        Creatinine,          Age, diagnoses,
   BP, temperature      WBC, Na, K...        admission, history

   trends +             abnormalities +      patient context +
   summaries            trends               comorbidities

          └─────────────────────┼─────────────────────┘
                                │
                                ▼
                     BUILD FEATURE VECTOR
                                │
                                ▼
                           ML MODEL
                                │
                                ▼
                  ICU MORTALITY PREDICTION
                         ex: 68%
                                │
                                ▼
                              SHAP
                                │
                                ▼
                     WHY DID ML PREDICT 68%?
                                │
                                ▼
                    OUR MODEL ASSESSMENT
                                │
                                │
                    ┌───────────┴───────────┐
                    │                       │
                    ▼                       ▼
             OUR ML PREDICTION        APACHE RESULT
                  68%                 e.g. 61%
                    │                       │
                    └───────────┬───────────┘
                                ▼
                         COMPARISON
                                │
                                ▼
                  agreement / disagreement
                                │
                                ▼
                   FINAL SEVERITY OUTPUT
"""

'\n                         SEVERITY AGENT\n                                │\n                                │ receives outputs from\n                                │ the 3 specialised agents\n                                │\n          ┌─────────────────────┼─────────────────────┐\n          │                     │                     │\n          ▼                     ▼                     ▼\n     VITAL OUTPUT          LAB OUTPUT          CLINICAL OUTPUT\n\n   HR, SpO₂, RR,        Creatinine,          Age, diagnoses,\n   BP, temperature      WBC, Na, K...        admission, history\n\n   trends +             abnormalities +      patient context +\n   summaries            trends               comorbidities\n\n          └─────────────────────┼─────────────────────┘\n                                │\n                                ▼\n                     BUILD FEATURE VECTOR\n                                │\n                                ▼\n                           ML MODEL\n

**NOTA:** O APACHE não confirma uma previsão individual, porque é apenas outro sistema de prognóstico e também pode errar.
A função dele será servir de baseline: comparar o desempenho global do nosso modelo com o APACHE no mesmo conjunto de teste, usando os outcomes reais como referência.

#### Informação que o *AI/LLM Agent* irá receber:

In [4]:
"""
                    EXPLANATION AGENT
                           │
                           │ receives
                           │
          ┌────────────────┼────────────────┐
          │                │                │
          ▼                ▼                ▼
   AGENT FINDINGS      ML PREDICTION      SHAP OUTPUT

 Vital findings       ICU mortality      Main features
 Lab findings         probability        driving prediction
 Clinical context
          │                │                │
          └────────────────┼────────────────┘
                           │
                           ▼
                 INTEGRATE INFORMATION
                           │
                           ▼
               GENERATE HUMAN-READABLE
                    EXPLANATION
                           │
                           ▼
                 STRUCTURED REPORT
"""

'\n                    EXPLANATION AGENT\n                           │\n                           │ receives\n                           │\n          ┌────────────────┼────────────────┐\n          │                │                │\n          ▼                ▼                ▼\n   AGENT FINDINGS      ML PREDICTION      SHAP OUTPUT\n\n Vital findings       ICU mortality      Main features\n Lab findings         probability        driving prediction\n Clinical context\n          │                │                │\n          └────────────────┼────────────────┘\n                           │\n                           ▼\n                 INTEGRATE INFORMATION\n                           │\n                           ▼\n               GENERATE HUMAN-READABLE\n                    EXPLANATION\n                           │\n                           ▼\n                 STRUCTURED REPORT\n'

#### Estrutura do *AI/LLM agente* que irá validar o que fizemos até agora:

In [5]:
"""
                     CRITIC AGENT
                           │
                           │ receives
                           │
          ┌────────────────┼────────────────┐
          │                │                │
          ▼                ▼                ▼
   ORIGINAL DATA      AGENT OUTPUTS     FINAL EXPLANATION

 Vital/Lab/Clinical   Vital findings    LLM-generated report
 patient data         Lab findings
                      Clinical findings
                      ML + SHAP
          │                │                │
          └────────────────┼────────────────┘
                           │
                           ▼
                    VALIDATE REPORT
                           │
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
   FACTUAL CHECK     CONSISTENCY CHECK   SUPPORT CHECK

 Is every claim      Does explanation    Are there claims
 supported by        match ML/SHAP       not present in
 patient data?       and agent outputs?  the available data?
          │                │                │
          └────────────────┼────────────────┘
                           │
                           ▼
                    ERROR OR OK?
                      ↙          ↘
                   ERROR          OK
                     │            │
                     ▼            ▼
               SEND FEEDBACK   FINAL REPORT
                     │
                     ▼
            EXPLANATION AGENT
                     │
                     └──────→ regenerate
"""

'\n                     CRITIC AGENT\n                           │\n                           │ receives\n                           │\n          ┌────────────────┼────────────────┐\n          │                │                │\n          ▼                ▼                ▼\n   ORIGINAL DATA      AGENT OUTPUTS     FINAL EXPLANATION\n\n Vital/Lab/Clinical   Vital findings    LLM-generated report\n patient data         Lab findings\n                      Clinical findings\n                      ML + SHAP\n          │                │                │\n          └────────────────┼────────────────┘\n                           │\n                           ▼\n                    VALIDATE REPORT\n                           │\n          ┌────────────────┼────────────────┐\n          ▼                ▼                ▼\n   FACTUAL CHECK     CONSISTENCY CHECK   SUPPORT CHECK\n\n Is every claim      Does explanation    Are there claims\n supported by        match ML/SHAP       not present in\n

### 1. Analizar os diferentes tipos de dados (sup)

#### 1.1 Pacientes

In [2]:
from pathlib import Path

DATA_DIR = Path("data")

In [3]:
import pandas as pd

patient = pd.read_csv(DATA_DIR /"patient.csv.gz")

print(patient.shape)
print(patient.columns.tolist())

patient.head()

(2520, 29)
['patientunitstayid', 'patienthealthsystemstayid', 'gender', 'age', 'ethnicity', 'hospitalid', 'wardid', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitoffset', 'hospitaladmitsource', 'hospitaldischargeyear', 'hospitaldischargetime24', 'hospitaldischargeoffset', 'hospitaldischargelocation', 'hospitaldischargestatus', 'unittype', 'unitadmittime24', 'unitadmitsource', 'unitvisitnumber', 'unitstaytype', 'admissionweight', 'dischargeweight', 'unitdischargetime24', 'unitdischargeoffset', 'unitdischargelocation', 'unitdischargestatus', 'uniquepid']


,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,...,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,141764,129391,Female,87,Caucasian,59,91,NaN,157.5,23:36:00,...,ICU to SDU,2,stepdown/other,NaN,NaN,18:58:00,344,Home,Alive,002-1039
1,141765,129391,Female,87,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",157.5,23:36:00,...,Emergency Department,1,admit,46.5,45.0,13:14:00,2250,Step-Down Unit (SDU),Alive,002-1039
2,143870,131022,Male,76,Caucasian,68,103,"Endarterectomy, carotid",167.0,20:46:00,...,Operating Room,1,admit,77.5,79.4,10:00:00,793,Floor,Alive,002-12289
3,144815,131736,Female,34,Caucasian,56,82,"Overdose, other toxin, poison or drug",172.7,01:44:00,...,Emergency Department,1,admit,60.3,60.7,20:48:00,1121,Other External,Alive,002-1116
4,145427,132209,Male,61,Caucasian,68,103,"GI perforation/rupture, surgery for",177.8,23:48:00,...,Operating Room,1,admit,91.7,93.1,22:47:00,1369,Floor,Alive,002-12243


In [7]:
patient.info()
patient.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2520 entries, 0 to 2519
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patientunitstayid          2520 non-null   int64  
 1   patienthealthsystemstayid  2520 non-null   int64  
 2   gender                     2516 non-null   object 
 3   age                        2516 non-null   object 
 4   ethnicity                  2481 non-null   object 
 5   hospitalid                 2520 non-null   int64  
 6   wardid                     2520 non-null   int64  
 7   apacheadmissiondx          2221 non-null   object 
 8   admissionheight            2451 non-null   float64
 9   hospitaladmittime24        2520 non-null   object 
 10  hospitaladmitoffset        2520 non-null   int64  
 11  hospitaladmitsource        1926 non-null   object 
 12  hospitaldischargeyear      2520 non-null   int64  
 13  hospitaldischargetime24    2520 non-null   objec

patientunitstayid               0
patienthealthsystemstayid       0
gender                          4
age                             4
ethnicity                      39
hospitalid                      0
wardid                          0
apacheadmissiondx             299
admissionheight                69
hospitaladmittime24             0
hospitaladmitoffset             0
hospitaladmitsource           594
hospitaldischargeyear           0
hospitaldischargetime24         0
hospitaldischargeoffset         0
hospitaldischargelocation      32
hospitaldischargestatus        28
unittype                        0
unitadmittime24                 0
unitadmitsource                22
unitvisitnumber                 0
unitstaytype                    0
admissionweight               198
dischargeweight              1284
unitdischargetime24             0
unitdischargeoffset             0
unitdischargelocation           4
unitdischargestatus             2
uniquepid                       0
dtype: int64

In [8]:
print(patient["unitdischargestatus"].value_counts(dropna=False))
print()

print(patient["hospitaldischargestatus"].value_counts(dropna=False))

unitdischargestatus
Alive      2392
Expired     126
NaN           2
Name: count, dtype: int64

hospitaldischargestatus
Alive      2280
Expired     212
NaN          28
Name: count, dtype: int64


In [9]:
# 1. Quantos outcomes temos?
print("ICU mortality:")
print(patient["unitdischargestatus"].value_counts(dropna=False))

print("\nHospital mortality:")
print(patient["hospitaldischargestatus"].value_counts(dropna=False))


# 2. Quantas PESSOAS diferentes temos?
print("\nICU stays:", patient["patientunitstayid"].nunique())
print("Unique patients:", patient["uniquepid"].nunique())


# 3. Que valores existem na idade?
print("\nAge:")
print(patient["age"].unique())

ICU mortality:
unitdischargestatus
Alive      2392
Expired     126
NaN           2
Name: count, dtype: int64

Hospital mortality:
hospitaldischargestatus
Alive      2280
Expired     212
NaN          28
Name: count, dtype: int64

ICU stays: 2520
Unique patients: 1841

Age:
['87' '76' '34' '61' '55' '60' '28' '> 89' '59' '44' '66' '41' '63' '57'
 '52' '23' '73' '39' '20' '29' '71' '18' '83' '84' '47' '38' '46' '49'
 '81' '68' '51' '17' '89' '85' '56' '80' '48' '74' '16' '64' '75' '67'
 '72' '82' '77' '78' '58' '88' '53' '54' '65' '69' '62' '50' '42' '32'
 '25' '40' '43' '79' '70' '86' '30' '37' '26' '19' '31' '21' '45' nan '22'
 '27' '33' '35' '24' '36' '15']


#### 1.2 Vitais

In [ ]:
vitals = pd.read_csv(DATA_DIR /"vitalPeriodic.csv.gz")

print(vitals.shape)
print(vitals.columns.tolist())
vitals.info()
vitals.head(10)

(1634960, 19)
['vitalperiodicid', 'patientunitstayid', 'observationoffset', 'temperature', 'sao2', 'heartrate', 'respiration', 'cvp', 'etco2', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'pasystolic', 'padiastolic', 'pamean', 'st1', 'st2', 'st3', 'icp']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1634960 entries, 0 to 1634959
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   vitalperiodicid    1634960 non-null  int64  
 1   patientunitstayid  1634960 non-null  int64  
 2   observationoffset  1634960 non-null  int64  
 3   temperature        112820 non-null   float64
 4   sao2               1441700 non-null  float64
 5   heartrate          1627894 non-null  float64
 6   respiration        1381351 non-null  float64
 7   cvp                202284 non-null   float64
 8   etco2              72780 non-null    float64
 9   systemicsystolic   226475 non-null   float64
 10  systemicdiastol

,vitalperiodicid,patientunitstayid,observationoffset,temperature,sao2,heartrate,respiration,cvp,etco2,systemicsystolic,systemicdiastolic,systemicmean,pasystolic,padiastolic,pamean,st1,st2,st3,icp
0,29524122,141765,1179,NaN,NaN,82.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29557845,141765,189,NaN,97.0,76.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,29524442,141765,1169,NaN,NaN,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,29513052,141765,1534,NaN,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,29524600,141765,1164,NaN,NaN,86.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,29558795,141765,159,NaN,96.0,80.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,29511454,141765,1584,NaN,NaN,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,29553160,141765,334,NaN,96.0,76.0,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,29485816,141764,74,NaN,NaN,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-1.7,-1.7,NaN
9,29476745,141764,344,NaN,NaN,104.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.1,-0.7,-0.7,NaN


In [11]:
vitals_24h = vitals[
    (vitals["observationoffset"] >= 0) &
    (vitals["observationoffset"] <= 1440)
]

In [12]:
vitals.head(20)

,vitalperiodicid,patientunitstayid,observationoffset,temperature,sao2,heartrate,respiration,cvp,etco2,systemicsystolic,systemicdiastolic,systemicmean,pasystolic,padiastolic,pamean,st1,st2,st3,icp
0,29524122,141765,1179,NaN,NaN,82.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29557845,141765,189,NaN,97.0,76.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,29524442,141765,1169,NaN,NaN,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,29513052,141765,1534,NaN,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,29524600,141765,1164,NaN,NaN,86.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,29558795,141765,159,NaN,96.0,80.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,29511454,141765,1584,NaN,NaN,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,29553160,141765,334,NaN,96.0,76.0,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,29485816,141764,74,NaN,NaN,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-1.7,-1.7,NaN
9,29476745,141764,344,NaN,NaN,104.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.1,-0.7,-0.7,NaN


In [13]:
# Vitals nas primeiras 24h
vitals_24h = vitals[
    (vitals["observationoffset"] >= 0) &
    (vitals["observationoffset"] <= 1440)
]

print("Vital observations first 24h:", len(vitals_24h))
print("ICU stays with vitals first 24h:",
      vitals_24h["patientunitstayid"].nunique())

vital_cols = [
    "temperature", "sao2", "heartrate", "respiration",
    "cvp", "etco2", "systemicsystolic",
    "systemicdiastolic", "systemicmean",
    "pasystolic", "padiastolic", "pamean", "icp"
]

print("\nCoverage by vital:")
for col in vital_cols:
    n_stays = vitals_24h.loc[
        vitals_24h[col].notna(),
        "patientunitstayid"
    ].nunique()

    print(f"{col}: {n_stays}/2520 ({n_stays/2520*100:.1f}%)")

Vital observations first 24h: 553846
ICU stays with vitals first 24h: 2370

Coverage by vital:
temperature: 152/2520 (6.0%)
sao2: 2320/2520 (92.1%)
heartrate: 2359/2520 (93.6%)
respiration: 2151/2520 (85.4%)
cvp: 251/2520 (10.0%)
etco2: 101/2520 (4.0%)
systemicsystolic: 399/2520 (15.8%)
systemicdiastolic: 398/2520 (15.8%)
systemicmean: 401/2520 (15.9%)
pasystolic: 81/2520 (3.2%)
padiastolic: 81/2520 (3.2%)
pamean: 81/2520 (3.2%)
icp: 7/2520 (0.3%)


#### 1.3 Laboratório

In [ ]:
labs = pd.read_csv(DATA_DIR /"lab.csv.gz")

print(labs.shape)
print(labs.columns.tolist())
labs.info()
labs.head(10)

(434660, 10)
['labid', 'patientunitstayid', 'labresultoffset', 'labtypeid', 'labname', 'labresult', 'labresulttext', 'labmeasurenamesystem', 'labmeasurenameinterface', 'labresultrevisedoffset']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 434660 entries, 0 to 434659
Data columns (total 10 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   labid                    434660 non-null  int64  
 1   patientunitstayid        434660 non-null  int64  
 2   labresultoffset          434660 non-null  int64  
 3   labtypeid                434660 non-null  int64  
 4   labname                  434660 non-null  object 
 5   labresult                431939 non-null  float64
 6   labresulttext            434639 non-null  object 
 7   labmeasurenamesystem     415754 non-null  object 
 8   labmeasurenameinterface  415763 non-null  object 
 9   labresultrevisedoffset   434660 non-null  int64  
dtypes: float64(1), int64(5), objec

,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,437880563,1754323,-647,3,Hct,38.30,38.3,%,%,-631
1,437880572,1754323,-647,3,platelets x 1000,181.00,181,K/mcL,k/mm cu,-631
2,437880560,1754323,-647,3,RBC,4.86,4.86,M/mcL,m/mm cu,-631
3,437880570,1754323,-647,3,-monos,8.70,8.7,%,%,-631
4,437880571,1754323,-647,3,MCHC,30.40,30.4,g/dL,%,-631
5,435917188,1754323,-207,3,WBC x 1000,14.30,14.3,K/mcL,k/mm cu,-175
6,437880567,1754323,-647,3,WBC x 1000,5.40,5.4,K/mcL,k/mm cu,-631
7,435917185,1754323,-207,3,Hct,32.50,32.5,%,%,-175
8,437880566,1754323,-647,3,Hgb,11.60,11.6,g/dL,g/dL,-631
9,435917183,1754323,-207,3,MPV,8.20,8.2,fL,fL,-175


In [15]:
labs["labname"].value_counts().head(50)
print("Número de exames diferentes:", labs["labname"].nunique())

Número de exames diferentes: 147


In [16]:
# Exames mais frequentes
print(labs["labname"].value_counts().head(50))

# Quantos exames diferentes existem
print("\nUnique lab tests:", labs["labname"].nunique())

# Quantos ICU stays têm pelo menos um lab nas primeiras 24h
labs_24h = labs[
    (labs["labresultoffset"] >= 0) &
    (labs["labresultoffset"] <= 1440)
]

print("\nLab results first 24h:", len(labs_24h))
print("ICU stays with labs first 24h:",
      labs_24h["patientunitstayid"].nunique())

labname
bedside glucose             29965
potassium                   16337
sodium                      15414
glucose                     14983
chloride                    14655
creatinine                  14631
BUN                         14563
Hgb                         14389
calcium                     14123
Hct                         14042
bicarbonate                 13636
platelets x 1000            12738
WBC x 1000                  12680
RBC                         12630
MCV                         12375
MCHC                        12366
MCH                         12093
RDW                         11639
anion gap                   11551
MPV                          8098
-lymphs                      8086
-monos                       8049
-eos                         7561
-polys                       7311
-basos                       7300
magnesium                    7138
albumin                      6433
AST (SGOT)                   5546
ALT (SGPT)                   5499
total 

In [17]:
lab_coverage = (
    labs_24h
    .dropna(subset=["labresult"])
    .groupby("labname")["patientunitstayid"]
    .nunique()
    .sort_values(ascending=False)
)

print(lab_coverage.head(50))

labname
creatinine                  2092
sodium                      2092
BUN                         2089
glucose                     2075
potassium                   2072
chloride                    2056
calcium                     2016
Hct                         2000
Hgb                         1970
WBC x 1000                  1956
bicarbonate                 1944
platelets x 1000            1920
RBC                         1897
MCV                         1873
MCHC                        1872
MCH                         1813
RDW                         1766
anion gap                   1644
MPV                         1234
-lymphs                     1234
-monos                      1232
bedside glucose             1199
magnesium                   1140
-eos                        1128
-polys                      1126
-basos                      1090
albumin                     1051
AST (SGOT)                   951
ALT (SGPT)                   941
total bilirubin              938
to

In [18]:
lab_coverage_pct = (lab_coverage / 2520 * 100).round(1)

print(lab_coverage_pct.head(50))

labname
creatinine                  83.0
sodium                      83.0
BUN                         82.9
glucose                     82.3
potassium                   82.2
chloride                    81.6
calcium                     80.0
Hct                         79.4
Hgb                         78.2
WBC x 1000                  77.6
bicarbonate                 77.1
platelets x 1000            76.2
RBC                         75.3
MCV                         74.3
MCHC                        74.3
MCH                         71.9
RDW                         70.1
anion gap                   65.2
MPV                         49.0
-lymphs                     49.0
-monos                      48.9
bedside glucose             47.6
magnesium                   45.2
-eos                        44.8
-polys                      44.7
-basos                      43.3
albumin                     41.7
AST (SGOT)                  37.7
ALT (SGPT)                  37.3
total bilirubin             37.2
to

#### 1.4 Diagnóstico

In [ ]:
diagnosis = pd.read_csv(DATA_DIR /"diagnosis.csv.gz")

print(diagnosis.shape)
print(diagnosis.columns.tolist())
diagnosis.info()
diagnosis.head(10)

(24978, 7)
['diagnosisid', 'patientunitstayid', 'activeupondischarge', 'diagnosisoffset', 'diagnosisstring', 'icd9code', 'diagnosispriority']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24978 entries, 0 to 24977
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   diagnosisid          24978 non-null  int64 
 1   patientunitstayid    24978 non-null  int64 
 2   activeupondischarge  24978 non-null  bool  
 3   diagnosisoffset      24978 non-null  int64 
 4   diagnosisstring      24978 non-null  object
 5   icd9code             21206 non-null  object
 6   diagnosispriority    24978 non-null  object
dtypes: bool(1), int64(3), object(3)
memory usage: 1.2+ MB


,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,7607199,346380,False,5028,cardiovascular|ventricular disorders|hypertension,"401.9, I10",Other
1,7570429,346380,False,685,neurologic|altered mental status / pain|change...,"780.09, R41.82",Major
2,7705483,346380,True,5035,cardiovascular|shock / hypotension|hypotension,"458.9, I95.9",Major
3,7848601,346380,True,5035,neurologic|altered mental status / pain|schizo...,"295.90, F20.9",Major
4,7451475,346380,False,5028,pulmonary|disorders of vasculature|pulmonary e...,"415.19, I26.99",Major
5,7839760,346380,False,5028,cardiovascular|shock / hypotension|hypotension,"458.9, I95.9",Major
6,7194271,346380,True,5035,renal|disorder of kidney|acute renal failure,"584.9, N17.9",Primary
7,6926969,346380,True,5035,general|other syndromes|syncope,"780.2, R55",Major
8,7517655,346380,False,5028,neurologic|misc|Parkinson's disease,"332.0, G20",Major
9,7251429,346380,False,5028,cardiovascular|shock / hypotension|dehydration,"276.51, E86.0",Major


In [20]:
diagnosis_24h = diagnosis[
    (diagnosis["diagnosisoffset"] >= 0) &
    (diagnosis["diagnosisoffset"] <= 1440)
]

print("Diagnoses first 24h:", len(diagnosis_24h))

print(
    "ICU stays with diagnosis first 24h:",
    diagnosis_24h["patientunitstayid"].nunique()
)

print("\nPriority:")
print(diagnosis_24h["diagnosispriority"].value_counts())

print("\nMost common diagnoses:")
print(diagnosis_24h["diagnosisstring"].value_counts().head(20))

Diagnoses first 24h: 12649
ICU stays with diagnosis first 24h: 2133

Priority:
diagnosispriority
Other      4884
Major      4735
Primary    3030
Name: count, dtype: int64

Most common diagnoses:
diagnosisstring
pulmonary|respiratory failure|acute respiratory failure                                                   343
endocrine|glucose metabolism|diabetes mellitus                                                            314
cardiovascular|ventricular disorders|congestive heart failure                                             303
renal|disorder of kidney|acute renal failure                                                              271
pulmonary|pulmonary infections|pneumonia                                                                  268
pulmonary|respiratory failure|acute respiratory distress                                                  249
neurologic|altered mental status / pain|change in mental status                                           246
cardiovascular|vasc

#### 1.5 Ver admissões no Historico hospitalar do paciente

In [ ]:
admission_dx = pd.read_csv(DATA_DIR /"admissionDx.csv.gz")

print("\n--- ADMISSION DX ---")
print(admission_dx.shape)
print(admission_dx.columns.tolist())
admission_dx.info()
display(admission_dx.head(10))


--- ADMISSION DX ---
(7578, 6)
['admissiondxid', 'patientunitstayid', 'admitdxenteredoffset', 'admitdxpath', 'admitdxname', 'admitdxtext']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7578 entries, 0 to 7577
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   admissiondxid         7578 non-null   int64 
 1   patientunitstayid     7578 non-null   int64 
 2   admitdxenteredoffset  7578 non-null   int64 
 3   admitdxpath           7578 non-null   object
 4   admitdxname           7578 non-null   object
 5   admitdxtext           7578 non-null   object
dtypes: int64(3), object(3)
memory usage: 355.3+ KB


,admissiondxid,patientunitstayid,admitdxenteredoffset,admitdxpath,admitdxname,admitdxtext
0,7351978,2900423,162,admission diagnosis|Non-operative Organ System...,Cardiovascular,Cardiovascular
1,7351977,2900423,162,admission diagnosis|Was the patient admitted f...,No,No
2,7351979,2900423,162,admission diagnosis|All Diagnosis|Non-operativ...,"Sepsis, pulmonary","Sepsis, pulmonary"
3,7745060,2902156,944,admission diagnosis|All Diagnosis|Non-operativ...,"Rhythm disturbance (atrial, supraventricular)","Rhythm disturbance (atrial, supraventricular)"
4,7745059,2902156,944,admission diagnosis|Non-operative Organ System...,Cardiovascular,Cardiovascular
5,7745058,2902156,944,admission diagnosis|Was the patient admitted f...,No,No
6,8251201,2903553,2,admission diagnosis|Non-operative Organ System...,Cardiovascular,Cardiovascular
7,8251200,2903553,2,admission diagnosis|Was the patient admitted f...,No,No
8,8251202,2903553,2,admission diagnosis|All Diagnosis|Non-operativ...,"Sepsis, other","Sepsis, other"
9,8250122,2903847,4,admission diagnosis|Non-operative Organ System...,Respiratory,Respiratory


In [ ]:
past_history = pd.read_csv(DATA_DIR /"pastHistory.csv.gz")

print("\n--- PAST HISTORY ---")
print(past_history.shape)
print(past_history.columns.tolist())
past_history.info()
display(past_history.head(10))


--- PAST HISTORY ---
(12109, 8)
['pasthistoryid', 'patientunitstayid', 'pasthistoryoffset', 'pasthistoryenteredoffset', 'pasthistorynotetype', 'pasthistorypath', 'pasthistoryvalue', 'pasthistoryvaluetext']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12109 entries, 0 to 12108
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   pasthistoryid             12109 non-null  int64 
 1   patientunitstayid         12109 non-null  int64 
 2   pasthistoryoffset         12109 non-null  int64 
 3   pasthistoryenteredoffset  12109 non-null  int64 
 4   pasthistorynotetype       12109 non-null  object
 5   pasthistorypath           12109 non-null  object
 6   pasthistoryvalue          12109 non-null  object
 7   pasthistoryvaluetext      12109 non-null  object
dtypes: int64(4), object(4)
memory usage: 756.9+ KB


,pasthistoryid,patientunitstayid,pasthistoryoffset,pasthistoryenteredoffset,pasthistorynotetype,pasthistorypath,pasthistoryvalue,pasthistoryvaluetext
0,990803,141765,7,12,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
1,970059,143870,4,10,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
2,1180401,144815,32,41,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
3,1194998,145427,8,13,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
4,896652,147307,53,56,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
5,1261966,147784,10,20,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,Performed,Performed
6,1261967,147784,10,20,Comprehensive Progress,notes/Progress Notes/Past History/Organ System...,medication dependent,medication dependent
7,1150985,148611,-93,-82,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
8,1252461,149713,4,9,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
9,1137517,151179,18,29,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,Performed,Performed


#### 1.6 Analisar APACHE

In [ ]:
apache_aps = pd.read_csv(DATA_DIR /"apacheApsVar.csv.gz")
apache_pred = pd.read_csv(DATA_DIR /"apachePredVar.csv.gz")
apache_result = pd.read_csv(DATA_DIR /"apachePatientResult.csv.gz")

for name, df in {
    "APACHE APS": apache_aps,
    "APACHE Prediction Variables": apache_pred,
    "APACHE Patient Result": apache_result
}.items():

    print("\n" + "="*50)
    print(name)
    print("="*50)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nInfo:")
    df.info()

    print("\nFirst rows:")
    display(df.head())


APACHE APS
Shape: (2205, 26)

Columns:
['apacheapsvarid', 'patientunitstayid', 'intubated', 'vent', 'dialysis', 'eyes', 'motor', 'verbal', 'meds', 'urine', 'wbc', 'temperature', 'respiratoryrate', 'sodium', 'heartrate', 'meanbp', 'ph', 'hematocrit', 'creatinine', 'albumin', 'pao2', 'pco2', 'bun', 'glucose', 'bilirubin', 'fio2']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2205 entries, 0 to 2204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   apacheapsvarid     2205 non-null   int64  
 1   patientunitstayid  2205 non-null   int64  
 2   intubated          2205 non-null   int64  
 3   vent               2205 non-null   int64  
 4   dialysis           2205 non-null   int64  
 5   eyes               2205 non-null   int64  
 6   motor              2205 non-null   int64  
 7   verbal             2205 non-null   int64  
 8   meds               2205 non-null   int64  
 9   urine              220

,apacheapsvarid,patientunitstayid,intubated,vent,dialysis,eyes,motor,verbal,meds,urine,...,ph,hematocrit,creatinine,albumin,pao2,pco2,bun,glucose,bilirubin,fio2
0,92788,141765,0,0,0,4,6,5,0,-1.0,...,-1.0,37.8,1.04,-1.0,-1.0,-1.0,28.0,61,-1.0,-1
1,8893,143870,0,0,0,4,6,5,0,-1.0,...,-1.0,34.1,1.14,-1.0,-1.0,-1.0,14.0,140,-1.0,-1
2,79585,144815,0,0,0,4,6,5,0,-1.0,...,-1.0,36.6,0.63,3.6,-1.0,-1.0,6.0,82,0.5,-1
3,203242,145427,0,0,0,4,6,5,0,-1.0,...,-1.0,40.4,1.05,-1.0,-1.0,-1.0,14.0,118,-1.0,-1
4,154681,147307,0,0,0,4,6,5,0,-1.0,...,-1.0,-1.0,-1.00,-1.0,-1.0,-1.0,-1.0,-1,-1.0,-1



APACHE Prediction Variables
Shape: (2205, 51)

Columns:
['apachepredvarid', 'patientunitstayid', 'sicuday', 'saps3day1', 'saps3today', 'saps3yesterday', 'gender', 'teachtype', 'region', 'bedcount', 'admitsource', 'graftcount', 'meds', 'verbal', 'motor', 'eyes', 'age', 'admitdiagnosis', 'thrombolytics', 'diedinhospital', 'aids', 'hepaticfailure', 'lymphoma', 'metastaticcancer', 'leukemia', 'immunosuppression', 'cirrhosis', 'electivesurgery', 'activetx', 'readmit', 'ima', 'midur', 'ventday1', 'oobventday1', 'oobintubday1', 'diabetes', 'managementsystem', 'var03hspxlos', 'pao2', 'fio2', 'ejectfx', 'creatinine', 'dischargelocation', 'visitnumber', 'amilocation', 'day1meds', 'day1verbal', 'day1motor', 'day1eyes', 'day1pao2', 'day1fio2']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2205 entries, 0 to 2204
Data columns (total 51 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   apachepredvarid    2205 non-null   int64  

,apachepredvarid,patientunitstayid,sicuday,saps3day1,saps3today,saps3yesterday,gender,teachtype,region,bedcount,...,creatinine,dischargelocation,visitnumber,amilocation,day1meds,day1verbal,day1motor,day1eyes,day1pao2,day1fio2
0,4004,141765,1,0,0,0,1,0,3,12,...,1.04,4,1,-1,0,5,6,4,-1.0,-1
1,8047,143870,1,0,0,0,0,0,3,14,...,1.14,4,1,-1,0,5,6,4,-1.0,-1
2,1798882,144815,1,0,0,0,1,0,3,8,...,0.63,8,1,-1,0,5,6,4,-1.0,-1
3,1485310,145427,1,0,0,0,0,0,3,14,...,1.05,4,1,-1,0,5,6,4,-1.0,-1
4,27991,147307,1,0,0,0,1,0,3,43,...,-1.00,4,1,-1,0,5,6,4,-1.0,-1



APACHE Patient Result
Shape: (3676, 23)

Columns:
['apachepatientresultsid', 'patientunitstayid', 'physicianspeciality', 'physicianinterventioncategory', 'acutephysiologyscore', 'apachescore', 'apacheversion', 'predictedicumortality', 'actualicumortality', 'predictediculos', 'actualiculos', 'predictedhospitalmortality', 'actualhospitalmortality', 'predictedhospitallos', 'actualhospitallos', 'preopmi', 'preopcardiaccath', 'ptcawithin24h', 'unabridgedunitlos', 'unabridgedhosplos', 'actualventdays', 'predventdays', 'unabridgedactualventdays']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3676 entries, 0 to 3675
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   apachepatientresultsid         3676 non-null   int64  
 1   patientunitstayid              3676 non-null   int64  
 2   physicianspeciality            3676 non-null   object 
 3   physicianinterventioncategory  367

,apachepatientresultsid,patientunitstayid,physicianspeciality,physicianinterventioncategory,acutephysiologyscore,apachescore,apacheversion,predictedicumortality,actualicumortality,predictediculos,...,predictedhospitallos,actualhospitallos,preopmi,preopcardiaccath,ptcawithin24h,unabridgedunitlos,unabridgedhosplos,actualventdays,predventdays,unabridgedactualventdays
0,31917,141765,hospitalist,Unknown,23,47,IV,0.008247,ALIVE,0.722231,...,2.881710,1.8222,0,0,0,1.5625,1.8222,NaN,NaN,NaN
1,31918,141765,hospitalist,Unknown,23,47,IVa,0.012311,ALIVE,1.374807,...,3.173925,1.8222,0,0,0,1.5625,1.8222,NaN,NaN,NaN
2,21398,143870,family practice,Unknown,43,60,IVa,0.015707,ALIVE,3.021671,...,6.032627,0.8465,0,0,0,0.5506,0.8465,NaN,NaN,NaN
3,21397,143870,family practice,Unknown,43,60,IV,0.017739,ALIVE,3.006522,...,5.955292,0.8465,0,0,0,0.5506,0.8465,NaN,NaN,NaN
4,181,144815,internal medicine,I,25,25,IV,0.001834,ALIVE,0.592446,...,2.196294,0.8063,0,0,0,0.7784,0.8063,NaN,NaN,NaN


---

### 2. Conversão de .csv.gz -> .csv

In [ ]:
import pandas as pd

files = [
    "patient",
    "vitalPeriodic",
    "vitalAperiodic",
    "lab",
    "diagnosis",
    "admissionDx",
    "pastHistory",
    "apacheApsVar",
    "apachePredVar",
    "apachePatientResult"
]

for file in files:
    print(f"A converter {file}.csv.gz...")

    df = pd.read_csv(DATA_DIR /f"{file}.csv.gz")
    df.to_csv(f"{file}.csv", index=False)

    print(f"✓ {file}.csv criado")

print("\nConversão concluída!")

A converter patient.csv.gz...
✓ patient.csv criado
A converter vitalPeriodic.csv.gz...
✓ vitalPeriodic.csv criado
A converter vitalAperiodic.csv.gz...
✓ vitalAperiodic.csv criado
A converter lab.csv.gz...
✓ lab.csv criado
A converter diagnosis.csv.gz...
✓ diagnosis.csv criado
A converter admissionDx.csv.gz...
✓ admissionDx.csv criado
A converter pastHistory.csv.gz...
✓ pastHistory.csv criado
A converter apacheApsVar.csv.gz...
✓ apacheApsVar.csv criado
A converter apachePredVar.csv.gz...
✓ apachePredVar.csv criado
A converter apachePatientResult.csv.gz...
✓ apachePatientResult.csv criado

Conversão concluída!


### 3. Análise mais profunda dos dados a usar

#### 3.1 Vitals

In [ ]:
import pandas as pd

# =========================
# 1. LER OS FICHEIROS
# =========================

files = {
    "vitalPeriodic": pd.read_csv(DATA_DIR /"vitalPeriodic.csv"),
    "vitalAperiodic": pd.read_csv(DATA_DIR /"vitalAperiodic.csv")
}


# =========================
# 2. FUNÇÃO DE ANÁLISE
# =========================

def analyze_dataset(name, df):

    print("\n")
    print("=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)

    # -------------------------
    # DIMENSÃO
    # -------------------------
    print("\n1. DIMENSION")
    print("-" * 50)
    print(f"Shape: {df.shape}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

    # -------------------------
    # NOMES DAS COLUNAS
    # -------------------------
    print("\n2. COLUMNS")
    print("-" * 50)

    for col in df.columns:
        print(f" - {col}")

    # -------------------------
    # TIPOS DE DADOS
    # -------------------------
    print("\n3. DATA TYPES")
    print("-" * 50)

    print(df.dtypes)

    # -------------------------
    # PRIMEIRAS LINHAS
    # -------------------------
    print("\n4. FIRST 5 ROWS")
    print("-" * 50)

    print(df.head())

    # -------------------------
    # VALORES EM FALTA
    # -------------------------
    print("\n5. MISSING VALUES")
    print("-" * 50)

    missing_count = df.isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100

    missing_table = pd.DataFrame({
        "Missing values": missing_count,
        "Missing (%)": missing_percentage.round(2)
    })

    print(missing_table)

    # -------------------------
    # VALORES ÚNICOS
    # -------------------------
    print("\n6. UNIQUE VALUES")
    print("-" * 50)

    print(df.nunique())

    # -------------------------
    # DUPLICADOS
    # -------------------------
    print("\n7. DUPLICATED ROWS")
    print("-" * 50)

    duplicates = df.duplicated().sum()

    print(f"Duplicated rows: {duplicates:,}")

    # -------------------------
    # ESTATÍSTICAS DESCRITIVAS
    # -------------------------
    print("\n8. DESCRIPTIVE STATISTICS")
    print("-" * 50)

    print(df.describe(include="all").T)

    # -------------------------
    # NÚMERO DE PACIENTES
    # -------------------------
    if "patientunitstayid" in df.columns:

        print("\n9. PATIENTS")
        print("-" * 50)

        n_patients = df["patientunitstayid"].nunique()

        print(f"Unique patientunitstayid: {n_patients:,}")

        records_per_patient = df.groupby(
            "patientunitstayid"
        ).size()

        print("\nRecords per patient:")
        print(records_per_patient.describe())

    # -------------------------
    # MEMORY USAGE
    # -------------------------
    print("\n10. MEMORY USAGE")
    print("-" * 50)

    memory_mb = df.memory_usage(
        deep=True
    ).sum() / (1024 ** 2)

    print(f"{memory_mb:.2f} MB")

In [26]:
analyze_dataset(
    "vitalPeriodic",
    files["vitalPeriodic"]
)

analyze_dataset(
    "vitalAperiodic",
    files["vitalAperiodic"]
)



DATASET: vitalPeriodic

1. DIMENSION
--------------------------------------------------
Shape: (1634960, 19)
Rows: 1,634,960
Columns: 19

2. COLUMNS
--------------------------------------------------
 - vitalperiodicid
 - patientunitstayid
 - observationoffset
 - temperature
 - sao2
 - heartrate
 - respiration
 - cvp
 - etco2
 - systemicsystolic
 - systemicdiastolic
 - systemicmean
 - pasystolic
 - padiastolic
 - pamean
 - st1
 - st2
 - st3
 - icp

3. DATA TYPES
--------------------------------------------------
vitalperiodicid        int64
patientunitstayid      int64
observationoffset      int64
temperature          float64
sao2                 float64
heartrate            float64
respiration          float64
cvp                  float64
etco2                float64
systemicsystolic     float64
systemicdiastolic    float64
systemicmean         float64
pasystolic           float64
padiastolic          float64
pamean               float64
st1                  float64
st2             

In [27]:
periodic_24h = files["vitalPeriodic"][
    (files["vitalPeriodic"]["observationoffset"] >= 0) &
    (files["vitalPeriodic"]["observationoffset"] <= 1440)
].copy()

aperiodic_24h = files["vitalAperiodic"][
    (files["vitalAperiodic"]["observationoffset"] >= 0) &
    (files["vitalAperiodic"]["observationoffset"] <= 1440)
].copy()

In [28]:
print("Periodic first 24h:")
print(periodic_24h.shape)

print("\nAperiodic first 24h:")
print(aperiodic_24h.shape)

print("\nPatients periodic:")
print(periodic_24h["patientunitstayid"].nunique())

print("\nPatients aperiodic:")
print(aperiodic_24h["patientunitstayid"].nunique())

Periodic first 24h:
(553846, 19)

Aperiodic first 24h:
(109601, 13)

Patients periodic:
2370

Patients aperiodic:
2315


In [29]:
def patient_coverage(df, variables):

    total_patients = df["patientunitstayid"].nunique()

    results = []

    for var in variables:

        # pacientes que têm pelo menos uma medição válida
        patients_with_value = (
            df.loc[df[var].notna(), "patientunitstayid"]
            .nunique()
        )

        coverage = (patients_with_value / total_patients) * 100

        results.append({
            "variable": var,
            "total_patients": total_patients,
            "patients_with_value": patients_with_value,
            "patients_without_value": total_patients - patients_with_value,
            "coverage_%": round(coverage, 2),
            "missing_patients_%": round(100 - coverage, 2)
        })

    return pd.DataFrame(results)

In [30]:
periodic_variables = [
    "heartrate",
    "sao2",
    "respiration",
    "temperature"
]

periodic_coverage = patient_coverage(
    periodic_24h,
    periodic_variables
)

display(periodic_coverage)

,variable,total_patients,patients_with_value,patients_without_value,coverage_%,missing_patients_%
0,heartrate,2370,2359,11,99.54,0.46
1,sao2,2370,2320,50,97.89,2.11
2,respiration,2370,2151,219,90.76,9.24
3,temperature,2370,152,2218,6.41,93.59


In [31]:
aperiodic_variables = [
    "noninvasivesystolic",
    "noninvasivediastolic",
    "noninvasivemean"
]

aperiodic_coverage = patient_coverage(
    aperiodic_24h,
    aperiodic_variables
)

display(aperiodic_coverage)

,variable,total_patients,patients_with_value,patients_without_value,coverage_%,missing_patients_%
0,noninvasivesystolic,2315,2310,5,99.78,0.22
1,noninvasivediastolic,2315,2310,5,99.78,0.22
2,noninvasivemean,2315,2311,4,99.83,0.17


#### 3.2 Laboratório

In [ ]:
import pandas as pd

# Load dataset
lab = pd.read_csv(DATA_DIR /"lab.csv")

print("=" * 100)
print("DATASET: lab")
print("=" * 100)

# 1. DIMENSION
print("\n1. DIMENSION")
print("-" * 50)
print("Shape:", lab.shape)
print(f"Rows: {lab.shape[0]:,}")
print(f"Columns: {lab.shape[1]}")

# 2. COLUMNS
print("\n2. COLUMNS")
print("-" * 50)
for col in lab.columns:
    print(f" - {col}")

# 3. DATA TYPES
print("\n3. DATA TYPES")
print("-" * 50)
print(lab.dtypes)

# 4. FIRST ROWS
print("\n4. FIRST ROWS")
print("-" * 50)
display(lab.head())

# 5. MISSING VALUES
print("\n5. MISSING VALUES")
print("-" * 50)

missing = pd.DataFrame({
    "missing_values": lab.isnull().sum(),
    "missing_percentage": (lab.isnull().mean() * 100).round(2)
})

display(
    missing[missing["missing_values"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

# 6. UNIQUE VALUES
print("\n6. UNIQUE VALUES")
print("-" * 50)

unique_values = pd.DataFrame({
    "unique_values": lab.nunique()
})

display(unique_values)

# 7. DUPLICATES
print("\n7. DUPLICATES")
print("-" * 50)
print("Duplicated rows:", lab.duplicated().sum())

# 8. NUMERIC DESCRIPTION
print("\n8. NUMERIC DESCRIPTION")
print("-" * 50)
display(lab.describe())

# 9. PATIENTS
print("\n9. PATIENTS")
print("-" * 50)
print(
    "Unique patient stays:",
    lab["patientunitstayid"].nunique()
)

# 10. LAB TESTS
print("\n10. LAB TESTS")
print("-" * 50)
print(
    "Number of different laboratory tests:",
    lab["labname"].nunique()
)

print("\nMost frequent laboratory tests:")
display(
    lab["labname"]
    .value_counts()
    .head(50)
    .to_frame("number_of_results")
)

# 11. LAB RESULT OFFSET
print("\n11. LAB RESULT OFFSET")
print("-" * 50)
display(lab["labresultoffset"].describe())

# 12. LAB RESULT
print("\n12. LAB RESULT")
print("-" * 50)
display(lab["labresult"].describe())

# 13. MEMORY USAGE
print("\n13. MEMORY USAGE")
print("-" * 50)
print(
    f"{lab.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

DATASET: lab

1. DIMENSION
--------------------------------------------------
Shape: (434660, 10)
Rows: 434,660
Columns: 10

2. COLUMNS
--------------------------------------------------
 - labid
 - patientunitstayid
 - labresultoffset
 - labtypeid
 - labname
 - labresult
 - labresulttext
 - labmeasurenamesystem
 - labmeasurenameinterface
 - labresultrevisedoffset

3. DATA TYPES
--------------------------------------------------
labid                        int64
patientunitstayid            int64
labresultoffset              int64
labtypeid                    int64
labname                     object
labresult                  float64
labresulttext               object
labmeasurenamesystem        object
labmeasurenameinterface     object
labresultrevisedoffset       int64
dtype: object

4. FIRST ROWS
--------------------------------------------------


,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,437880563,1754323,-647,3,Hct,38.30,38.3,%,%,-631
1,437880572,1754323,-647,3,platelets x 1000,181.00,181,K/mcL,k/mm cu,-631
2,437880560,1754323,-647,3,RBC,4.86,4.86,M/mcL,m/mm cu,-631
3,437880570,1754323,-647,3,-monos,8.70,8.7,%,%,-631
4,437880571,1754323,-647,3,MCHC,30.40,30.4,g/dL,%,-631



5. MISSING VALUES
--------------------------------------------------


,missing_values,missing_percentage
labmeasurenamesystem,18906,4.35
labmeasurenameinterface,18897,4.35
labresult,2721,0.63
labresulttext,21,0.00



6. UNIQUE VALUES
--------------------------------------------------


,unique_values
labid,434660
patientunitstayid,2444
labresultoffset,21247
labtypeid,6
labname,147
labresult,5561
labresulttext,7668
labmeasurenamesystem,31
labmeasurenameinterface,341
labresultrevisedoffset,24633


... (5 outputs left out)

count    434660.000000
mean       4533.898516
std        8827.101453
min      -68394.000000
25%          69.000000
50%        1993.000000
75%        6428.000000
max       90072.000000
Name: labresultoffset, dtype: float64


12. LAB RESULT
--------------------------------------------------


count    431939.000000
mean         59.324990
std         317.910478
min         -30.000000
25%           5.600000
50%          19.600000
75%          88.000000
max       93450.000000
Name: labresult, dtype: float64


13. MEMORY USAGE
--------------------------------------------------
108.25 MB
